# 호출 한도와 비용 상한

호출 전에 예상 비용을 예약해 한도 밖 요청을 모델로 보내지 않습니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** reserve()는 다음 호출의 예상 횟수와 계획 비용을 현재 사용량에 더해 상한과 비교합니다. 통과하면 예약값을 갱신하고, 넘으면 모델 invoke() 전에 예외를 반환합니다. 호출 뒤 usage_metadata는 실제 사용량으로 별도 기록하며 예약액을 실제 청구액으로 바꾸지 않습니다.

**다른 상황에 적용.** 한 질문이 재작성·평가·생성 세 호출을 요구한다면 시작 전에 세 호출의 최악 계획을 예약할 수 있습니다. 남은 범위가 부족하면 검색 결과만 보여 주거나 추가 기능을 생략하는 축소 경로를 선택합니다.

**비교와 경계.** 호출 횟수 제한과 제공자 rate limit은 다릅니다. 전자는 애플리케이션이 정한 세션 정책이고 후자는 일정 시간의 API 허용량입니다. 이 절의 0.002달러는 계산 연습용 값이며 실제 가격 정보가 아닙니다.

**이 셀의 코드 읽기.** 이 셀은 앞 셀에서 만든 값을 입력으로 읽어 위에서 아래 순서로 처리하고, 왼쪽 변수에 다음 셀이 사용할 결과를 저장합니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

교육용 예상 비용을 호출당 0.002달러로 둡니다. 이 값은 실제 가격표가 아닙니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [2]:
from pathlib import Path
import os, sys
from dotenv import find_dotenv, load_dotenv

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part4/ch16-guardrails").is_dir())
LAB = ROOT / "labs/part4/ch16-guardrails"
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
print("Python:", sys.version.split()[0], "| API 키 준비: True")

Python: 3.12.13 | API 키 준비: True


### 다음 코드 셀 읽기

**이 셀의 코드 읽기.** `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4-mini", temperature=0, max_tokens=140,
    timeout=30, max_retries=0,
)
print("모델 준비:", type(model).__name__)

모델 준비: ChatOpenAI


## 기본 실행

예산은 호출 수와 금액을 각각 검사합니다.

**이 셀의 코드 읽기.** 이 셀을 실행하면 `CallBudget` 클래스 정의가 준비됩니다. 클래스 본문은 정의 시 처리되지만 인스턴스 값은 나중에 객체를 만들 때 생깁니다. `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `@dataclass`는 아래 클래스의 필드로 초기화 메서드 등을 만들어 주는 데코레이터입니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
from dataclasses import dataclass
import math

@dataclass
class CallBudget:
    max_calls: int
    max_cost_usd: float
    calls: int = 0
    cost_usd: float = 0.0

    def reserve(self, upper_bound_usd: float) -> None:
        valid = math.isfinite(upper_bound_usd) and upper_bound_usd >= 0
        if not valid or self.calls + 1 > self.max_calls:
            raise ValueError("호출 한도를 초과합니다.")
        if self.cost_usd + upper_bound_usd > self.max_cost_usd:
            raise ValueError("계획 비용 상한을 초과합니다.")
        self.calls += 1
        self.cost_usd += upper_bound_usd

budget = CallBudget(max_calls=2, max_cost_usd=0.005)
estimated = 0.002
budget.reserve(estimated)
answer = model.invoke("LLM 애플리케이션의 가드레일을 소프트웨어 관점에서 한 문장으로 정의하세요. '보장'이라는 표현은 쓰지 마세요.")
print(answer.text)
print("예약 상태:", budget.calls, f"${budget.cost_usd:.3f}")

LLM 애플리케이션의 가드레일은 모델의 입력·출력·행동을 소프트웨어적으로 제어해, 정책에 맞지 않거나 위험한 결과가 나오지 않도록 감시·필터링·완화하는 보호 계층입니다.
예약 상태: 1 $0.002


## 사용량 관찰

실제 반환된 사용량을 따로 읽어 예상값과 구분합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [5]:
usage = answer.usage_metadata or {}
print("입력 토큰:", usage.get("input_tokens"))
print("출력 토큰:", usage.get("output_tokens"))
print("예상 비용은 수업용 상한 계산값")

입력 토큰: 44
출력 토큰: 63
예상 비용은 수업용 상한 계산값


## 실패 재현

호출 수가 남아 있어도 금액 상한을 넘으면 예약을 거부합니다.

**이 셀의 코드 읽기.** `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `try` 안의 작업이 예외를 내면 `except` 블록이 그 실패를 받아 화면에 확인 가능한 결과를 남깁니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [6]:
# 예약이 실패하면 model.invoke()를 실행하지 않습니다.
try:
    budget.reserve(0.004)
except ValueError as error:
    print(type(error).__name__, error)

ValueError 계획 비용 상한을 초과합니다.


## 수정

요청을 축소해 남은 범위 안에서 예약합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [7]:
budget.reserve(0.001)
print("최종 상태:", budget.calls, f"${budget.cost_usd:.3f}")
assert budget.calls == 2 and budget.cost_usd <= budget.max_cost_usd

최종 상태: 2 $0.003
